# Peregrine · registered T4 baseline
Select a T4 runtime. Replace `<RUN_COMMIT>` with the pinned commit before running cell 1.


In [ ]:
!git clone https://github.com/ASKoshelenko/peregrine
%cd peregrine
!git checkout <RUN_COMMIT>


In [ ]:
!pip install -e '.[ml]' 'dvc[gs]'


In [ ]:
from google.colab import auth
auth.authenticate_user()


In [ ]:
!dvc pull data/processed/warehouse.dvc


In [ ]:
import os
import subprocess
os.environ['WANDB_API_KEY'] = subprocess.check_output(['gcloud', 'secrets', 'versions', 'access', 'latest', '--secret=peregrine-wandb-api-key', '--project=peregrine-edge-mlops'], text=True).strip()


In [ ]:
!PYTHONPATH=src python -m peregrine.cli train --data data/processed/warehouse/data.yaml --run-dir artifacts/runs/baseline --override run.name=peregrine-t4-baseline-001


In [ ]:
from pathlib import Path
best = next(Path('artifacts/runs/baseline').rglob('best.pt'))
!PYTHONPATH=src python -m peregrine.cli eval --weights {best} --data data/processed/warehouse/data.yaml --split test --target x86_onnx_fp32 --out artifacts/real/eval-x86_onnx_fp32.json --device 0


In [ ]:
!pip freeze > artifacts/runs/baseline/pip-freeze.txt
!gsutil -m cp -r artifacts/runs/baseline artifacts/real gs://peregrine-edge-mlops-artifacts/runs/colab-t4-baseline-001/
